# Notebook 3 – Evaluating Load Surrogate Models

In this notebook we analyse and compare different surrogate models that predict a
single structural-load channel (e.g. power, tower base DEL, blade root DEL) for
the IEA 3.4 MW turbine.

We assume that:

- The **training/validation split** has already been created in Notebook 1.
- The **surrogate models** (NN, XGBoost, GPR, spline) have been trained and
  saved in Notebook 2 as `.pkl` bundles.

Our goals are:

1. **Visualise model behaviour** over wind speed and turbulence intensity.
2. **Compare models quantitatively** using several error metrics on the test set.
3. **Relate differences between models** to our domain knowledge about loads
   (smoothness, monotonicity, extrapolation, etc.).


In [ ]:
# CELL 1: Imports and configuration

import os
import pickle
import numpy as np
import pandas as pd

import xgboost as xgb

import plotly.graph_objects as go
from scipy.interpolate import RectBivariateSpline   # <- NEW

import warnings
from sklearn.exceptions import InconsistentVersionWarning

DATA_DIR = "Data"

# Choose regime and target to analyze in this notebook
SELECTED_REGIME = "Normal"      # "Normal", "Idling", "Shut", "Start"
TARGET_COL      = "TBFA_DEL4"  # e.g. "PowAc_mean", "TBFA_DEL4","TBSS_DEL4", "BRFW_DEL10","BREW_DEL10", "TTTOR_DEL4","MSBMZ_loc_DEL4" ...

# Input columns that models expect (Notebook 2 used these by default)
INPUT_COLS = [
    "Wind Speed",
    "Turbulence Intensity",
]

# Model definitions:
#   - name: string identifier used in plots / tables
#   - path: path to the pickle bundle (as saved in Notebook 2)
#   - For spline, use name "spline_interpolation" and path "" (empty string)
#     and this notebook will build the spline model from training data.
MODEL_DEFINITIONS = [
     ("xgb", r"Pretrained_models/Normal__TBFA_DEL4__xgb_optuna__rmse_minmax_log_300tr.pkl"),
     ("nn", r"Pretrained_models/Normal__TBFA_DEL4__nn_optuna__rmse_standard_iter150.pkl"),
     ("gpr", r"Pretrained_models/Normal__TBFA_DEL4__gpr_RBF+Matern32__rmse_minmax.pkl"),
     # ("my_model_nn","student/Normal__TBFA_DEL4__gpr_RBF+Matern32__my_model.pkl"), # Example of a custom model added by the user

    # Uncomment to include spline interpolation as a "model":
     ("spline_interpolation", ""),
]

# Shared WS / TI domain for sweeps (1D) and surfaces (3D)
# We will use these min/max/step to build np.arange(...) internally.
WSP_MIN, WSP_MAX, WSP_STEP = 4.0, 25.0, 0.1
TI_MIN,  TI_MAX,  TI_STEP = 4, 25, 0.5

# Fixed points for 1D sweeps
FIXED_TI_VALUES  = [6,12]         # for WS sweeps at fixed TI
FIXED_WSP_VALUES = [8.0, 18.0]    # for TI sweeps at fixed WS

# GPR uncertainty band in 1D sweeps (mean ± 2σ)
SHOW_GPR_UNCERTAINTY = False

# Default plot size for Plotly figures (used as convenience in call cells)
DEFAULT_PLOT_WIDTH  = 900
DEFAULT_PLOT_HEIGHT = 500


## 1. Load data and build a spline reference model

In this section we:

- Load the **full-factorial development data** and the **randomized test data**
  for the selected operating regime.
- Select the **target channel** we want to analyse.
- Build a **2D cubic spline interpolator** of the development data
  in the (Wind Speed, Turbulence Intensity) plane.

The spline acts as a **smooth reference model**:

- It is not a machine-learning surrogate, but a deterministic interpolation
  of the simulation grid.
- For this low-dimensional, low-noise, full-factorial dataset,
  the spline is a strong baseline to compare the ML models against.


In [ ]:
# CELL 2: Load training (full factorial) and test (randomized) data

train_file = os.path.join(DATA_DIR, f"{SELECTED_REGIME}_training_data.csv")
test_file  = os.path.join(DATA_DIR, f"{SELECTED_REGIME}_test_data.csv")

if not os.path.exists(train_file):
    raise FileNotFoundError(f"Training file not found: {train_file}")
if not os.path.exists(test_file):
    raise FileNotFoundError(f"Test file not found: {test_file}")

df_train_full = pd.read_csv(train_file)
df_test       = pd.read_csv(test_file)

print(f"Loaded regime: {SELECTED_REGIME}")
print("  Development data (full factorial):", df_train_full.shape)
print("  Untouched test set (randomized):  ", df_test.shape)

# Quick sanity check
if TARGET_COL not in df_train_full.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in training data.")
if TARGET_COL not in df_test.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in test data.")

for col in INPUT_COLS:
    if col not in df_train_full.columns:
        raise ValueError(f"Input column '{col}' not in training data.")
    if col not in df_test.columns:
        raise ValueError(f"Input column '{col}' not in test data.")


In [ ]:
# DELL 3:  Helper: build cubic spline interpolator with nearest-neighbour fill 

def build_spline_interpolator(df, wsp_col, ti_col, target_col):
    """
    Build a 2D cubic spline interpolator for a (WS, TI) grid using SciPy's
    RectBivariateSpline.

    The DOE is allowed to be "almost" full factorial, i.e. some (WS, TI)
    combinations may be missing (typically at the edges where simulations
    were unstable and removed).

    Strategy:
      1) Build the full pivot table Z[TI, WS].
      2) For any NaN cell, fill it with the value of the nearest available
         (WS, TI) combination (nearest neighbour in physical WS/TI space).
      3) Use the filled Z to build a cubic spline over the full grid.

    Returns:
        predict_fn(X): X is an array of shape (n_samples, 2) with columns
                       [Wind Speed, Turbulence Intensity].
                       Returns y_pred of shape (n_samples,).
    """
    # Unique sorted grid values
    wsp_vals = np.sort(df[wsp_col].unique())
    ti_vals  = np.sort(df[ti_col].unique())

    # Pivot -> Z[ti_index, wsp_index]
    Z_pivot = df.pivot(index=ti_col, columns=wsp_col, values=target_col)

    # Reindex to ensure we cover the full WS/TI grid in sorted order
    Z_pivot = Z_pivot.reindex(index=ti_vals, columns=wsp_vals)
    Z = Z_pivot.values.copy()

    # Identify missing cells
    nan_mask = np.isnan(Z)
    n_missing = int(nan_mask.sum())
    if n_missing > 0:
        print(f"Filling {n_missing} missing (WS, TI) grid cells by nearest neighbour.")

        # Coordinates of valid cells
        valid_indices = np.argwhere(~nan_mask)   # (j, i) pairs where data exists
        valid_ti_vals = ti_vals[valid_indices[:, 0]]
        valid_ws_vals = wsp_vals[valid_indices[:, 1]]
        valid_values  = Z[~nan_mask]

        # Fill each missing cell
        missing_indices = np.argwhere(nan_mask)
        for j, i in missing_indices:
            ti_miss = ti_vals[j]
            ws_miss = wsp_vals[i]

            # Euclidean distance in (WS, TI) space
            d2 = (valid_ws_vals - ws_miss) ** 2 + (valid_ti_vals - ti_miss) ** 2
            k = np.argmin(d2)

            Z[j, i] = valid_values[k]

    # Sanity check: no NaNs should remain
    if np.isnan(Z).any():
        raise ValueError(
            "build_spline_interpolator: NaNs remain in grid after nearest-neighbour fill."
        )

    print(
        f"Spline grid: {len(wsp_vals)} WS points (from {wsp_vals[0]} to {wsp_vals[-1]}) x "
        f"{len(ti_vals)} TI points (from {ti_vals[0]} to {ti_vals[-1]})"
    )

    # Build cubic spline over the full regular grid.
    # RectBivariateSpline expects (y, x, z) i.e. first axis = TI, second = WS.
    spline = RectBivariateSpline(ti_vals, wsp_vals, Z, kx=3, ky=3)

    def predict_fn(X):
        """
        X: array-like, shape (n_samples, 2)
           columns [Wind Speed, Turbulence Intensity]

        Returns:
            y_pred: array, shape (n_samples,)
        """
        X = np.asarray(X)
        if X.ndim == 1:
            X = X.reshape(-1, 2)

        wsp = X[:, 0]
        ti  = X[:, 1]

        # Clip to grid range to avoid wild extrapolation
        wsp_clipped = np.clip(wsp, wsp_vals[0], wsp_vals[-1])
        ti_clipped  = np.clip(ti,  ti_vals[0],  ti_vals[-1])

        # Evaluate elementwise at (ti_i, wsp_i)
        y_pred = spline(ti_clipped, wsp_clipped, grid=False)
        return np.asarray(y_pred).ravel()

    return predict_fn


## 2. Load trained models and define a common prediction interface

Here we prepare a simple **model registry** and helper functions:

- We define a dictionary that maps a short **model name** (e.g. `"gpr"`,
  `"nn_grid"`, `"xgb_optuna"`, `"spline_interpolation"`) to the corresponding
  saved bundle on disk.
- Each bundle contains:
  - the fitted model (NN, XGBoost, or GPR),
  - the input and output scalers (including a custom `LogScaler`),
  - some metadata (e.g. label string for plots).

We also implement a unified `predict_with_entry(...)` function so that,
later on, we can evaluate **all models with the same call signature**:

```python
y_pred, y_std = predict_with_entry(entry, X, return_std=True/False)


In [ ]:
# CELL 4: Model loading helpers (LogScaler, unified interface, load from definitions)

class LogScaler:
    """
    Simple log-based scaler for positive or non-positive targets.

    This matches the behaviour used when training the models in Notebook 2,
    so that pickled LogScaler instances can be unpickled correctly.

    - During fit, it finds a shift so that y + shift > 0 for all samples.
    - transform:  y -> log(y + shift)
    - inverse_transform: log_y -> exp(log_y) - shift
    """
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.shift_ = None

    def fit(self, y):
        y = np.asarray(y)
        min_y = np.min(y)
        if min_y <= 0:
            self.shift_ = -min_y + self.eps
        else:
            self.shift_ = 0.0
        return self

    def transform(self, y):
        if self.shift_ is None:
            raise RuntimeError("LogScaler not fitted")
        y = np.asarray(y)
        return np.log(y + self.shift_)

    def fit_transform(self, y):
        return self.fit(y).transform(y)

    def inverse_transform(self, y_log):
        if self.shift_ is None:
            raise RuntimeError("LogScaler not fitted")
        y_log = np.asarray(y_log)
        return np.exp(y_log) - self.shift_


def load_models_from_definitions(model_defs, df_train, wsp_col, ti_col, target_col):
    """
    Build a dictionary of model entries from MODEL_DEFINITIONS.

    Each entry has:
        - 'name'   : model name key from MODEL_DEFINITIONS
        - 'label'  : same as name (students can choose descriptive names)
        - 'type'   : 'spline' or 'pickle'
        - 'model'  : underlying model object (for pickle-based models)
        - 'x_scaler', 'y_scaler': may be None
        - 'input_cols': list of input columns used by the model
        - 'predict_fn': callable(X) -> y_pred (for spline; optional for others)
    """
    models = {}

    # First check if we need the spline interpolator
    need_spline = any(name == "spline_interpolation" for name, _ in model_defs)
    spline_predict_fn = None
    if need_spline:
        spline_predict_fn = build_spline_interpolator(
            df_train,
            wsp_col=wsp_col,
            ti_col=ti_col,
            target_col=target_col,
        )

    for name, path in model_defs:
        if name == "spline_interpolation":
            # Spline model built from df_train
            models[name] = {
                "name": name,
                "label": "Spline interpolation",
                "type": "spline",
                "model": None,
                "x_scaler": None,
                "y_scaler": None,
                "input_cols": [wsp_col, ti_col],
                "predict_fn": spline_predict_fn,
            }
            print(f"Registered spline model: {name}")
            continue

        if not os.path.exists(path):
            print(f"WARNING: Model file not found for '{name}': {path}")
            continue

        with open(path, "rb") as f:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", InconsistentVersionWarning)
                bundle = pickle.load(f)

        model    = bundle.get("model", None)
        x_scaler = bundle.get("x_scaler", None)
        y_scaler = bundle.get("y_scaler", None)
        meta     = bundle.get("meta", {})

        if model is None:
            print(f"WARNING: No 'model' in bundle for '{name}', skipping.")
            continue

        # Check regime/target consistency if stored
        regime = meta.get("regime", None)
        target = meta.get("target", None)
        if (regime is not None) and (regime != SELECTED_REGIME):
            print(f"Skipping '{name}': regime mismatch ({regime} != {SELECTED_REGIME}).")
            continue
        if (target is not None) and (target != TARGET_COL):
            print(f"Skipping '{name}': target mismatch ({target} != {TARGET_COL}).")
            continue

        input_cols = meta.get("input_cols", INPUT_COLS)

        models[name] = {
            "name": name,
            "label": name,  # if you want nicer labels, just choose nicer names
            "type": meta.get("model_type", "pickle"),
            "model": model,
            "x_scaler": x_scaler,
            "y_scaler": y_scaler,
            "input_cols": input_cols,
            "predict_fn": None,  # for spline only; others use 'model'
        }

        print(f"Loaded model '{name}' from {path}")
        print(f"  type      : {models[name]['type']}")
        print(f"  input_cols: {input_cols}")

    return models


def predict_with_entry(model_entry, X, return_std=False):
    """
    Unified prediction interface for all models.

    - For spline entries:
        uses model_entry['predict_fn'] directly on X.

    - For pickle-based models (NN, XGB, GPR):
        uses model_entry['model'], applies x_scaler / y_scaler if present.

    If return_std=True and the underlying model is a GPy GPRegression,
    returns (y_pred, y_std). Otherwise returns (y_pred, None).
    """
    # Spline model
    if model_entry["type"] == "spline":
        pred = model_entry["predict_fn"](X)
        if return_std:
            return pred, None
        else:
            return pred

    model    = model_entry["model"]
    x_scaler = model_entry["x_scaler"]
    y_scaler = model_entry["y_scaler"]

    X = np.asarray(X)
    # Expect X as shape (n_samples, n_features)
    if X.ndim == 1:
        X = X.reshape(-1, len(model_entry["input_cols"]))

    # Apply X scaler
    if x_scaler is not None:
        X_scaled = x_scaler.transform(X)
    else:
        X_scaled = X

    # Predict
    pred = model.predict(X_scaled)

    # GPy GPRegression: predict returns (mean, var)
    if isinstance(pred, (tuple, list)) and len(pred) == 2:
        mean_scaled, var_scaled = pred
        mean_scaled = mean_scaled.ravel()
        var_scaled  = var_scaled.ravel()

        if y_scaler is not None:
            mean = y_scaler.inverse_transform(mean_scaled.reshape(-1, 1)).ravel()
            # NOTE: variance transformation through a nonlinear inverse
            # (e.g. log) is not exact; here we keep std in scaled space
            # as a qualitative proxy.
            std = np.sqrt(var_scaled)
        else:
            mean = mean_scaled
            std  = np.sqrt(var_scaled)

        if return_std:
            return mean, std
        else:
            return mean

    # sklearn / xgboost: single array
    y_pred_scaled = np.asarray(pred).ravel()
    if y_scaler is not None:
        y_pred = y_scaler.inverse_transform(
            y_pred_scaled.reshape(-1, 1)
        ).ravel()
    else:
        y_pred = y_pred_scaled

    if return_std:
        return y_pred, None
    else:
        return y_pred


# Actually load the models now
MODELS = load_models_from_definitions(
    MODEL_DEFINITIONS,
    df_train=df_train_full,
    wsp_col="Wind Speed",
    ti_col="Turbulence Intensity",
    target_col=TARGET_COL,
)

print("\nModels registered for this notebook:")
for name, entry in MODELS.items():
    print("  ", name, "->", entry["label"], "| type:", entry["type"])


## 3. 1D sweeps over wind speed and turbulence intensity

In this section we:

- Sweep **wind speed** over a user-defined range for a few fixed TI levels.
- Sweep **turbulence intensity** for a few fixed wind speeds.

For each sweep we:

- Evaluate all selected models on the same grid,
- Plot the curves together (plus the spline, if enabled).

This helps us check whether:

- The trends make sense physically (e.g. loads vs WS shape),
- The models are smooth or show artefacts,
- Different models diverge in certain regions of the input space.


In [ ]:
# CELL 5: 1D sweep plotting functions (WS and TI) using Plotly

def make_grid(min_val, max_val, step):
    """
    Helper to build a 1D grid including the upper bound (approximately),
    using np.arange.
    """
    return np.arange(min_val, max_val + 0.5 * step, step)



def plot_ws_sweeps(
    models,
    fixed_ti_values,
    wsp_min,
    wsp_max,
    wsp_step,
    model_names=None,
    width=None,
    height=None,
    show_gpr_uncertainty=SHOW_GPR_UNCERTAINTY,
):
    """
    Plot target vs Wind Speed for several fixed TI values.

    For each TI in fixed_ti_values:
      - build a WS grid [wsp_min, wsp_max] with given step,
      - evaluate all selected models along this line,
      - show one Plotly figure per TI, with one curve per model.

    If show_gpr_uncertainty=True and a model is a GPR (type string contains 'gpr'),
    we also plot mean ± 2σ as dashed lines for that model.
    """
    if model_names is None:
        model_names = list(models.keys())

    wsp_grid = make_grid(wsp_min, wsp_max, wsp_step)

    for ti0 in fixed_ti_values:
        X_grid = np.column_stack([wsp_grid, np.full_like(wsp_grid, ti0)])
        fig = go.Figure()

        for name in model_names:
            if name not in models:
                print(f"WARNING: model '{name}' not found in MODELS dict, skipping.")
                continue

            entry = models[name]
            model_type_str = str(entry.get("type", "")).lower()
            is_gpr = "gpr" in model_type_str

            if show_gpr_uncertainty and is_gpr:
                # GPR: plot mean and ±2σ as dashed lines
                y_mean, y_std = predict_with_entry(entry, X_grid, return_std=True)

                if y_std is not None:
                    upper = y_mean + 2.0 * y_std
                    lower = y_mean - 2.0 * y_std

                    # Mean
                    fig.add_trace(go.Scatter(
                        x=wsp_grid,
                        y=y_mean,
                        mode="lines",
                        name=entry["label"],
                    ))
                    # +2σ
                    fig.add_trace(go.Scatter(
                        x=wsp_grid,
                        y=upper,
                        mode="lines",
                        name=f"{entry['label']} + 2σ",
                        line=dict(dash="dot"),
                        showlegend=True,
                    ))
                    # -2σ
                    fig.add_trace(go.Scatter(
                        x=wsp_grid,
                        y=lower,
                        mode="lines",
                        name=f"{entry['label']} - 2σ",
                        line=dict(dash="dot"),
                        showlegend=True,
                    ))
                else:
                    # Fallback if std is not available
                    fig.add_trace(go.Scatter(
                        x=wsp_grid,
                        y=y_mean,
                        mode="lines",
                        name=entry["label"],
                    ))

            else:
                # Non-GPR (or uncertainty disabled): just plot mean prediction
                y_pred = predict_with_entry(entry, X_grid, return_std=False)
                fig.add_trace(go.Scatter(
                    x=wsp_grid,
                    y=y_pred,
                    mode="lines",
                    name=entry["label"],
                ))

        fig.update_layout(
            title=f"{TARGET_COL} vs Wind Speed (TI = {ti0:.3f})",
            xaxis_title="Wind Speed [m/s]",
            yaxis_title=TARGET_COL,
            template="plotly_white",
        )

        # Apply size if provided
        if width is not None or height is not None:
            fig.update_layout(
                width=width if width is not None else DEFAULT_PLOT_WIDTH,
                height=height if height is not None else DEFAULT_PLOT_HEIGHT,
            )

        fig.show()

def plot_ti_sweeps(
    models,
    fixed_wsp_values,
    ti_min,
    ti_max,
    ti_step,
    model_names=None,
    width=None,
    height=None,
    show_gpr_uncertainty=SHOW_GPR_UNCERTAINTY,
):
    """
    Plot target vs Turbulence Intensity for several fixed WS values.

    For each WS in fixed_wsp_values:
      - build a TI grid [ti_min, ti_max] with given step,
      - evaluate all selected models along this line,
      - show one Plotly figure per WS, with one curve per model.

    If show_gpr_uncertainty=True and a model is a GPR (type string contains 'gpr'),
    we also plot mean ± 2σ as dashed lines for that model.
    """
    if model_names is None:
        model_names = list(models.keys())

    ti_grid = make_grid(ti_min, ti_max, ti_step)

    for ws0 in fixed_wsp_values:
        X_grid = np.column_stack([np.full_like(ti_grid, ws0), ti_grid])
        fig = go.Figure()

        for name in model_names:
            if name not in models:
                print(f"WARNING: model '{name}' not found in MODELS dict, skipping.")
                continue

            entry = models[name]
            model_type_str = str(entry.get("type", "")).lower()
            is_gpr = "gpr" in model_type_str

            if show_gpr_uncertainty and is_gpr:
                y_mean, y_std = predict_with_entry(entry, X_grid, return_std=True)

                if y_std is not None:
                    upper = y_mean + 2.0 * y_std
                    lower = y_mean - 2.0 * y_std

                    # Mean
                    fig.add_trace(go.Scatter(
                        x=ti_grid,
                        y=y_mean,
                        mode="lines",
                        name=entry["label"],
                    ))
                    # +2σ
                    fig.add_trace(go.Scatter(
                        x=ti_grid,
                        y=upper,
                        mode="lines",
                        name=f"{entry['label']} + 2σ",
                        line=dict(dash="dot"),
                        showlegend=True,
                    ))
                    # -2σ
                    fig.add_trace(go.Scatter(
                        x=ti_grid,
                        y=lower,
                        mode="lines",
                        name=f"{entry['label']} - 2σ",
                        line=dict(dash="dot"),
                        showlegend=True,
                    ))
                else:
                    fig.add_trace(go.Scatter(
                        x=ti_grid,
                        y=y_mean,
                        mode="lines",
                        name=entry["label"],
                    ))

            else:
                y_pred = predict_with_entry(entry, X_grid, return_std=False)
                fig.add_trace(go.Scatter(
                    x=ti_grid,
                    y=y_pred,
                    mode="lines",
                    name=entry["label"],
                ))

        fig.update_layout(
            title=f"{TARGET_COL} vs Turbulence Intensity (WS = {ws0:.2f} m/s)",
            xaxis_title="Turbulence Intensity [%]",  # or "[-]" depending on how you present it
            yaxis_title=TARGET_COL,
            template="plotly_white",
        )

        if width is not None or height is not None:
            fig.update_layout(
                width=width if width is not None else DEFAULT_PLOT_WIDTH,
                height=height if height is not None else DEFAULT_PLOT_HEIGHT,
            )

        fig.show()



In [ ]:
# CELL 6: Run 1D sweeps for the selected models

# Choose which models to include in the sweeps.
# By default, take all models that were loaded:
MODEL_NAMES_TO_PLOT = list(MODELS.keys())
print("Models used in sweeps:", MODEL_NAMES_TO_PLOT)

# You can also restrict manually, e.g.:
# MODEL_NAMES_TO_PLOT = ["spline_interpolation", "nn_grid", "xgb_grid", "gpr_RBF+RQ"]

# --- Wind Speed sweeps at fixed TI values ---
plot_ws_sweeps(
    models=MODELS,
    fixed_ti_values=FIXED_TI_VALUES,
    wsp_min=WSP_MIN,
    wsp_max=WSP_MAX,
    wsp_step=WSP_STEP,
    model_names=MODEL_NAMES_TO_PLOT,
    width=DEFAULT_PLOT_WIDTH,
    height=DEFAULT_PLOT_HEIGHT,
    show_gpr_uncertainty=SHOW_GPR_UNCERTAINTY,
)

# --- TI sweeps at fixed WS values ---
plot_ti_sweeps(
    models=MODELS,
    fixed_wsp_values=FIXED_WSP_VALUES,
    ti_min=TI_MIN,
    ti_max=TI_MAX,
    ti_step=TI_STEP,
    model_names=MODEL_NAMES_TO_PLOT,
    width=DEFAULT_PLOT_WIDTH,
    height=DEFAULT_PLOT_HEIGHT,
    show_gpr_uncertainty=SHOW_GPR_UNCERTAINTY,
)
#Spline and GPR respect the smooth, physical structure of the DOE pretty well.
#XGBoost is very powerful but piecewise-constant under the hood, so unless we constrain it with physics or monotonicity,
#it can produce jagged, slightly non-physical curves that still look OK in terms of global error metrics.


## 4. Scatter: model predictions vs test data (WS on x-axis)

Here we look at the **raw test points**:

- x-axis: Wind Speed from the test set,
- y-axis: target channel.

We then overlay the **predictions from each model** evaluated at the same
test inputs. This gives a quick visual answer to:

- Do the models follow the overall trend of the test data?
- Are some models systematically above/below the test cloud?
- Are there regions in wind speed where a model clearly misbehaves?

You can toggle individual models on/off by clicking on the legend entries.


In [ ]:
# CELL 7: Scatter: model responses overlaid on test-set scatter (vs Wind Speed)

def plot_models_vs_test_scatter_wsp(
    models,
    model_names,
    df_test,
    target_col,
    width=None,
    height=None,
    test_marker_size=9,
    model_marker_size=7,
):
    """
    Plot the surrogate responses overlaid on the test-set response
    as a function of Wind Speed.

    - x-axis: Wind Speed [m/s]
    - y-axis: target_col
    - one scatter for the true test values
    - one scatter per model evaluated at X_test

    Parameters
    ----------
    models : dict
        MODELS dict: name -> bundle (with 'predict' or ('model', 'x_scaler', 'y_scaler', ...)).
    model_names : list of str
        Names (keys in MODELS) to include.
    df_test : pd.DataFrame
        Test set DataFrame (must contain 'Wind Speed', target_col, and INPUT_COLS).
    target_col : str
        Name of the target column (e.g. TARGET_COL).
    width, height : int or None
        Figure size in pixels. If None, default sizes are used.
    test_marker_size : int or float
        Marker size for the test data scatter.
    model_marker_size : int or float
        Marker size for each model scatter.
    """
    # Inputs / outputs for the test set
    X_test = df_test[INPUT_COLS].values
    y_test = df_test[target_col].values
    wsp_test = df_test["Wind Speed"].values

    fig = go.Figure()

    # --- Test set scatter (black, slightly larger) ---
    fig.add_trace(go.Scattergl(
        x=wsp_test,
        y=y_test,
        mode="markers",
        name="Test data",
        marker=dict(size=test_marker_size, opacity=0.8, color="black"),
    ))

    # Define a cycle of marker symbols for the models
    symbol_cycle = [
        "circle",
        "square",
        "triangle-up",
        "diamond",
        "cross",
        "x",
        "triangle-down",
        "star",
    ]

    # --- Surrogate responses ---
    for i, name in enumerate(model_names):
        if name not in models:
            print(f"WARNING: model '{name}' not found in MODELS dict, skipping.")
            continue

        entry = models[name]
        # Unified prediction interface (same as in sweeps)
        y_pred = predict_with_entry(entry, X_test, return_std=False)

        symbol = symbol_cycle[i % len(symbol_cycle)]
        label = entry.get("label", name)

        fig.add_trace(go.Scattergl(
            x=wsp_test,
            y=y_pred,
            mode="markers",
            name=label,
            marker=dict(
                size=model_marker_size,
                opacity=0.7,
                symbol=symbol,
            ),
        ))

    fig.update_layout(
        title=f"{target_col} vs Wind Speed – test data and model predictions",
        xaxis_title="Wind Speed [m/s]",
        yaxis_title=target_col,
        template="plotly_white",
        legend_title_text="Trace",
    )

    if width is not None or height is not None:
        fig.update_layout(
            width=width if width is not None else DEFAULT_PLOT_WIDTH,
            height=height if height is not None else DEFAULT_PLOT_HEIGHT,
        )

    fig.show()


In [ ]:
# CELL 8: Overlay models vs test-set scatter (vs Wind Speed)

SCATTER_MODEL_NAMES = list(MODELS.keys())   # or e.g. ["spline_interpolation", "gpr", "nn_optuna", "xgb_optuna"]

print("Models included in WS–target scatter:", SCATTER_MODEL_NAMES)

plot_models_vs_test_scatter_wsp(
    models=MODELS,
    model_names=SCATTER_MODEL_NAMES,
    df_test=df_test,
    target_col=TARGET_COL,
    width=DEFAULT_PLOT_WIDTH+250,
    height=DEFAULT_PLOT_HEIGHT+300,
    test_marker_size=10,   # adjust here
    model_marker_size=10,   # and here
)


## 5. 2D response surfaces over the (WS, TI) plane

Next we visualise each model as a **2D surface** over the (Wind Speed,
Turbulence Intensity) plane:

- We create a regular grid in WS and TI,
- Evaluate each model on this grid,
- Plot an interactive 3D surface.

These plots show:

- Where loads are high or low across the operating space,
- How smooth each model is,
- And whether any model has strange behaviour away from the training points
  (e.g. oscillations, unrealistic ridges).

Each model gets its **own figure**, which you can rotate, zoom, and inspect.


In [ ]:
# CELL 9: 3D WS–TI surface plots for each model

def plot_surfaces_for_models(
    models,
    model_names,
    wsp_min,
    wsp_max,
    n_wsp,
    ti_min,
    ti_max,
    n_ti,
    width=None,
    height=None,
):
    """
    For each selected model, plot a 3D surface of TARGET_COL as a function
    of Wind Speed and Turbulence Intensity.

    The domain and resolution are controlled *only* by the arguments of this
    function (no hidden globals):
      - WS in [wsp_min, wsp_max], with n_wsp points
      - TI in [ti_min, ti_max], with n_ti points

    Parameters
    ----------
    models : dict
        MODELS dict: name -> bundle with 'predict' or ('model', 'x_scaler', 'y_scaler', 'type', 'label').
    model_names : list of str
        Names (keys in MODELS) to include.
    wsp_min, wsp_max : float
        Range of wind speed [m/s].
    n_wsp : int
        Number of grid points in WS direction.
    ti_min, ti_max : float
        Range of turbulence intensity (same units as in the data, here [%]).
    n_ti : int
        Number of grid points in TI direction.
    width, height : int or None
        Figure size in pixels. If None, uses DEFAULT_PLOT_* values.
    """
    # 1D grids
    wsp_grid = np.linspace(wsp_min, wsp_max, n_wsp)
    ti_grid  = np.linspace(ti_min,  ti_max,  n_ti)

    # 2D meshgrid for evaluation
    WSP, TI = np.meshgrid(wsp_grid, ti_grid)   # shapes: (n_ti, n_wsp)

    # Flatten for model evaluation
    X_flat = np.column_stack([WSP.ravel(), TI.ravel()])

    for name in model_names:
        if name not in models:
            print(f"WARNING: model '{name}' not found in MODELS dict, skipping.")
            continue

        entry = models[name]

        # Use the same unified interface as in the sweeps
        y_pred_flat = predict_with_entry(entry, X_flat, return_std=False)
        Z = y_pred_flat.reshape(TI.shape)   # (n_ti, n_wsp)

        fig = go.Figure(data=[
            go.Surface(
                x=wsp_grid,
                y=ti_grid,
                z=Z,
                colorscale="Viridis",
                showscale=True,
                colorbar=dict(title=TARGET_COL),
            )
        ])

        fig.update_layout(
            title=f"{TARGET_COL} surface – model: {entry.get('label', name)}",
            scene=dict(
                xaxis_title="Wind Speed [m/s]",
                yaxis_title="Turbulence Intensity [%]",
                zaxis_title=TARGET_COL,
            ),
            template="plotly_white",
        )

        if width is not None or height is not None:
            fig.update_layout(
                width=width if width is not None else DEFAULT_PLOT_WIDTH,
                height=height if height is not None else DEFAULT_PLOT_HEIGHT,
            )

        fig.show()


In [ ]:
# CELL 10: Run 3D WS–TI surfaces for selected models

# Choose which models to include in the surfaces
SURFACE_MODEL_NAMES = list(MODELS.keys())
print("Models used in 3D surfaces:", SURFACE_MODEL_NAMES)

# Define WS/TI domain and resolution for the surfaces
WSP_MIN_SURF, WSP_MAX_SURF, N_WSP_SURF = 3.0, 26.0, 40   # m/s
TI_MIN_SURF,  TI_MAX_SURF,  N_TI_SURF  = 4.0, 31.0, 30   # TI in [%]

plot_surfaces_for_models(
    models=MODELS,
    model_names=SURFACE_MODEL_NAMES,
    wsp_min=WSP_MIN_SURF,
    wsp_max=WSP_MAX_SURF,
    n_wsp=N_WSP_SURF,
    ti_min=TI_MIN_SURF,
    ti_max=TI_MAX_SURF,
    n_ti=N_TI_SURF,
    width=DEFAULT_PLOT_WIDTH+200,
    height=DEFAULT_PLOT_HEIGHT+200,
)


## 6. Global performance metrics on the test set

Now we compute **scalar performance metrics** for each model on the test set:

- ME   – mean error (bias),
- MAE  – mean absolute error,
- MSE  – mean squared error,
- RMSE – root mean squared error,
- MAPE – mean absolute percentage error,
- R²   – coefficient of determination,
- nRMSE_mean – RMSE normalised by the mean of the target,
- nRMSE_std  – RMSE normalised by the standard deviation of the target.

We collect these into a table and apply per-column colour coding:

- For each metric (column), darker shading = better value.
- This allows a quick comparison of models **metric by metric**.

This table is a compact summary of global performance; the plots that follow
help explain *why* some models do better than others.


In [ ]:
# CELL 11: Metrics on the test set for all models

from sklearn.metrics import r2_score

def compute_error_metrics(y_true, y_pred):
    """
    Compute scalar error metrics between y_true and y_pred.

    Returns a dict with:
      - ME        : mean error (bias)
      - MAE       : mean absolute error
      - MSE       : mean squared error
      - RMSE      : root mean squared error
      - MAPE      : mean absolute percentage error [%]
      - R2        : coefficient of determination
      - nRMSE_mean: RMSE normalized by mean(|y_true|)
      - nRMSE_std : RMSE normalized by std(y_true)
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    err = y_pred - y_true
    me  = np.mean(err)
    mae = np.mean(np.abs(err))
    mse = np.mean(err**2)
    rmse = np.sqrt(mse)

    # Avoid division by zero in MAPE
    denom = np.maximum(np.abs(y_true), 1e-6)
    mape = np.mean(np.abs(err) / denom) * 100.0

    r2 = r2_score(y_true, y_pred)

    nrmse_mean = rmse / np.mean(np.abs(y_true))
    nrmse_std  = rmse / np.std(y_true, ddof=0)

    return {
        "ME": me,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "nRMSE_mean": nrmse_mean,
        "nRMSE_std": nrmse_std,
    }


# Compute predictions and metrics for all models on the test set

X_test = df_test[INPUT_COLS].values
y_test = df_test[TARGET_COL].values

MODEL_METRICS   = {}
MODEL_PREDICTED = {}   # store predictions for later plots

for name, entry in MODELS.items():
    # Use the same unified prediction interface as in sweeps
    y_pred = predict_with_entry(entry, X_test, return_std=False)

    MODEL_PREDICTED[name] = y_pred
    MODEL_METRICS[name]   = compute_error_metrics(y_test, y_pred)

# Convert to DataFrame for a nice overview
metrics_df = pd.DataFrame(MODEL_METRICS).T

# print(f"Test-set metrics for target: {TARGET_COL}")
# display(metrics_df.sort_values("RMSE"))


# %% Color-coded metrics table (best-to-worst *per column*)

def style_metrics_table(metrics_df, metric_order=None):
    """
    Return a pandas Styler that color-codes metrics_df so that
    darker color = better value *within each column*.

    - For error metrics (ME, MAE, MSE, RMSE, MAPE, nRMSE_*): smaller is better.
    - For R2: larger is better.
    """
    df = metrics_df.copy()

    # Optional: reorder columns for display
    if metric_order is not None:
        df = df[metric_order]

    higher_better = {"R2"}   # everything else treated as "lower is better"

    styler = df.style

    # Apply a separate gradient per column
    for col in df.columns:
        vals = df[col].astype(float).copy()

        # Transform so that "better" = larger value for coloring
        if col == "ME":
            vals = -np.abs(vals)   # bias closer to zero is better
        elif col not in higher_better:
            vals = -vals           # smaller error is better

        # background_gradient will normalize this column independently
        # gmap must be a Series here (not a DataFrame)
        styler = styler.background_gradient(
            cmap="Blues",
            subset=[col],
            gmap=vals,
        )

    return styler


METRIC_ORDER = ["ME", "MAE", "MSE", "RMSE", "MAPE", "R2", "nRMSE_mean", "nRMSE_std"]

styled_metrics = style_metrics_table(metrics_df, metric_order=METRIC_ORDER)
styled_metrics

## 7. Parity plots (predicted vs true values)

Parity plots compare the model predictions directly to the true test values:

- x-axis: true target,
- y-axis: predicted target.

We add:

- A 1:1 line (perfect predictions),
- A fitted regression line (trend of predictions).

Good models:

- Cluster tightly around the 1:1 line,
- Show little systematic bias (no clear tilt away from the diagonal),
- Cover the full range of loads correctly.

We use the **same axis limits** for all models to make visual comparison easier.


In [ ]:
# CELL 12: Parity (one-to-one) plot for a single model

from sklearn.linear_model import LinearRegression

def plot_parity_for_model(
    model_name,
    y_true,
    y_pred,
    width=None,
    height=None,
):
    """
    Parity plot for a single model:

      - scatter of predicted vs true,
      - perfect y = x line,
      - fitted linear regression line y_pred = a + b * y_true.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Fit linear relation y_pred ≈ a + b * y_true
    lr = LinearRegression()
    lr.fit(y_true.reshape(-1, 1), y_pred)
    a = lr.intercept_
    b = lr.coef_[0]

    # Range for lines
    y_min = float(min(y_true.min(), y_pred.min()))
    y_max = float(max(y_true.max(), y_pred.max()))

    x_line = np.array([y_min, y_max])
    y_perfect = x_line               # perfect y = x
    y_fit     = a + b * x_line       # fitted line

    fig = go.Figure()

    # Scatter of predictions
    fig.add_trace(go.Scatter(
        x=y_true,
        y=y_pred,
        mode="markers",
        name=f"{model_name} predictions",
        marker=dict(size=6, opacity=0.6),
    ))

    # Perfect line
    fig.add_trace(go.Scatter(
        x=x_line,
        y=y_perfect,
        mode="lines",
        name="Perfect y = x",
        line=dict(dash="dash"),
    ))

    # Fitted line
    fig.add_trace(go.Scatter(
        x=x_line,
        y=y_fit,
        mode="lines",
        name=f"Fitted line (a + b·y)",
        line=dict(dash="dot"),
    ))

    fig.update_layout(
        title=f"Parity plot for {model_name} – {TARGET_COL}",
        xaxis_title=f"True {TARGET_COL}",
        yaxis_title=f"Predicted {TARGET_COL}",
        template="plotly_white",
    )

    if width is not None or height is not None:
        fig.update_layout(
            width=width if width is not None else DEFAULT_PLOT_WIDTH,
            height=height if height is not None else DEFAULT_PLOT_HEIGHT,
        )

    fig.show()
# %% Parity plots for multiple models (one figure per model)

def plot_parity_for_models(
    model_names=None,
    width=None,
    height=None,
):
    """
    Make a parity plot (predicted vs true) for each selected model.

    Parameters
    ----------
    model_names : list of str or None
        Model names (keys of MODEL_PREDICTED / MODELS).
        If None, use all models found in MODEL_PREDICTED.
    """
    if model_names is None:
        model_names = list(MODEL_PREDICTED.keys())

    for name in model_names:
        if name not in MODEL_PREDICTED:
            print(f"Model '{name}' not found in MODEL_PREDICTED, skipping.")
            continue

        print(f"\n=== Parity plot for model: {name} ===")
        plot_parity_for_model(
            model_name=name,
            y_true=y_test,
            y_pred=MODEL_PREDICTED[name],
            width=width,
            height=height,
        )


In [ ]:
# CELL 13: Parity plots for all selected models

MODELS_FOR_PARITY = list(MODEL_PREDICTED.keys())   # or e.g. ["gpr", "xgb_optuna", "nn_optuna", "spline_interpolation"]

plot_parity_for_models(
    model_names=MODELS_FOR_PARITY,
    width=DEFAULT_PLOT_WIDTH,
    height=DEFAULT_PLOT_HEIGHT,
)


## 8. Build point-wise error arrays for each model

To analyse errors in more detail, we first compute, for each test sample:

- `error` = y_pred − y_true                      (signed error),
- `pe`    = (y_pred − y_true) / |y_true| · 100   (signed percentage error),
- `ape`   = |error| / |y_true| · 100             (absolute percentage error).

We store these arrays in a dictionary `MODEL_ERRORS[model_name]` so that the
following plots (boxplots, histograms, error vs WS/TI, heatmaps) can all
reuse the same error data.


In [ ]:
# CELL 14: Build error arrays for each model on the test set

# We already have:
# X_test, y_test, MODEL_PREDICTED
MODEL_ERRORS = {}

for name, y_pred in MODEL_PREDICTED.items():
    err = y_pred - y_test                      # signed error

    denom = np.maximum(np.abs(y_test), 1e-6)
    pe  = err / denom * 100.0                  # signed percentage error [%]
    ape = np.abs(err) / denom * 100.0          # absolute percentage error [%]

    MODEL_ERRORS[name] = {
        "error": err,
        "pe":    pe,
        "ape":   ape,
    }

print("Stored error arrays for models:", list(MODEL_ERRORS.keys()))


## 9. Error distributions: boxplots and histograms

We now look at the **distribution** of errors for each model:

- **Boxplots** per model:
  - Show median, quartiles, and outliers.
  - Useful to compare spread and robustness.

- **Histograms** with all models overlaid:
  - Show how errors are centred (or biased),
  - Use a vertical line at 0 for context.

We can visualise:

- `error`  (signed error),
- `pe`     (signed percentage error),
- `ape`    (absolute percentage error).

This helps us see, for example, if a model has a small RMSE but a long tail of
large errors, or if a model is slightly biased but very tight otherwise.


In [ ]:
# CELL 15: Boxplots of errors and absolute percentage errors (all selected models)

def plot_error_boxplots(
    model_errors,
    which="ape",
    model_names=None,
    width=None,
    height=None,
):
    """
    Boxplot of errors for multiple models.

    Parameters
    ----------
    model_errors : dict
        MODEL_ERRORS dict: model_name -> {'error': ..., 'ape': ...}.
    which : {"error", "ape"}
        - "error": use signed errors (y_pred - y_true).
        - "ape": use absolute percentage error [%].
    model_names : list of str or None
        Models to include. If None, use all keys of model_errors.
    """
   

    if model_names is None:
        model_names = list(model_errors.keys())

    if which not in {"error", "pe", "ape"}:
        raise ValueError("which must be 'error', 'pe', or 'ape'")

    if which == "error":
        label = "Error (y_pred - y_true)"
    elif which == "pe":
        label = "Percentage error [%]"
    else:  # "ape"
        label = "Absolute percentage error [%]"

    title = f"Boxplot of {label} for different models (target: {TARGET_COL})"

    fig = go.Figure()

    for name in model_names:
        if name not in model_errors:
            print(f"Model '{name}' not in MODEL_ERRORS, skipping.")
            continue

        vals = model_errors[name][which]

        fig.add_trace(go.Box(
            y=vals,
            name=name,
            boxmean=True,
        ))

    fig.update_layout(
        title=title,
        yaxis_title=label,
        xaxis_title="Model",
        template="plotly_white",
    )

    if width is not None or height is not None:
        fig.update_layout(
            width=width if width is not None else DEFAULT_PLOT_WIDTH,
            height=height if height is not None else DEFAULT_PLOT_HEIGHT,
        )

    fig.show()


In [ ]:
# CELL 16: Run boxplots for errors and APE

MODELS_FOR_BOX = list(MODEL_ERRORS.keys())   # or e.g. ["gpr", "xgb_optuna", "nn_optuna", "spline_interpolation"]

# Boxplot of signed errors
plot_error_boxplots(
    model_errors=MODEL_ERRORS,
    which="error",
    model_names=MODELS_FOR_BOX,
    width=DEFAULT_PLOT_WIDTH,
    height=DEFAULT_PLOT_HEIGHT,
)

# Boxplot of absolute percentage errors
plot_error_boxplots(
    model_errors=MODEL_ERRORS,
    which="ape",
    model_names=MODELS_FOR_BOX,
    width=DEFAULT_PLOT_WIDTH,
    height=DEFAULT_PLOT_HEIGHT,
)

# Boxplot of absolute percentage errors
plot_error_boxplots(
    model_errors=MODEL_ERRORS,
    which="pe",
    model_names=MODELS_FOR_BOX,
    width=DEFAULT_PLOT_WIDTH,
    height=DEFAULT_PLOT_HEIGHT,
)

In [ ]:
# CELL 17: Histograms of errors / absolute percentage errors

def plot_error_histograms(
    model_errors,
    which="error",
    model_names=None,
    nbins=40,
    width=None,
    height=None,
):
    """
    Overlaid histograms of errors for multiple models.

    Parameters
    ----------
    model_errors : dict
        MODEL_ERRORS dict: model_name -> {'error': ..., 'ape': ...}.
    which : {"error", "ape"}
        - "error": signed error (y_pred - y_true)
        - "ape": absolute percentage error [%]
        - "pe":    signed percentage error [%]
    model_names : list of str or None
        Models to include. If None, use all keys of model_errors.
    nbins : int
        Number of histogram bins.
    """
    if model_names is None:
        model_names = list(model_errors.keys())

    if which not in {"error", "pe", "ape"}:
        raise ValueError("which must be 'error', 'pe', or 'ape'")

    if which == "error":
        label = "Error (y_pred - y_true)"
    elif which == "pe":
        label = "Percentage error [%]"
    else:
        label = "Absolute percentage error [%]"

    title = f"Histogram of {label} for different models (target: {TARGET_COL})"

    fig = go.Figure()

    # Collect all values first to get a common range
    all_vals = []
    for name in model_names:
        if name not in model_errors:
            print(f"Model '{name}' not in MODEL_ERRORS, skipping.")
            continue
        all_vals.append(model_errors[name][which])

    all_vals = np.concatenate(all_vals)
    vmin, vmax = np.min(all_vals), np.max(all_vals)

    for name in model_names:
        if name not in model_errors:
            continue
        vals = model_errors[name][which]

        fig.add_trace(go.Histogram(
            x=vals,
            name=name,
            nbinsx=nbins,
            histnorm="probability",
            opacity=0.5,
        ))

    fig.update_layout(
        title=title,
        xaxis_title=label,
        yaxis_title="Probability",
        template="plotly_white",
        barmode="overlay",
    )

    # Vertical reference line at 0
    fig.add_vline(
        x=0.0,
        line_dash="dash",
        line_color="black",
        annotation_text="0",
        annotation_position="top",
    )

    if width is not None or height is not None:
        fig.update_layout(
            width=width if width is not None else DEFAULT_PLOT_WIDTH,
            height=height if height is not None else DEFAULT_PLOT_HEIGHT,
        )

    fig.show()

In [ ]:
# CELL 18: Run error histograms

MODELS_FOR_HIST = list(MODEL_ERRORS.keys())   # or subset

# Signed error histogram
plot_error_histograms(
    model_errors=MODEL_ERRORS,
    which="error",   #  "error" or "ape" if you want absolute percentage error or "pe" for percentage error
    model_names=MODELS_FOR_HIST,
    nbins=40,
    width=DEFAULT_PLOT_WIDTH,
    height=DEFAULT_PLOT_HEIGHT,
)



## 10. Error vs target, wind speed, and turbulence intensity

Next we investigate how errors depend on:

- The **magnitude of the target** (are large loads harder to predict?),
- The **wind speed**,
- The **turbulence intensity**.

For each model we can plot:

- `error`, `pe`, or `ape` vs:
  - true target,
  - WS,
  - TI.

This helps answer questions like:

- Does the model systematically underpredict high loads?
- Is performance worse at low or high wind speeds?
- Are errors larger at specific turbulence intensities?


In [ ]:
# CELL 19: Scatter of error vs chosen x-variable (one figure per model)

def plot_error_vs_x_for_model(
    model_name,
    model_errors,
    which="error",
    x_source="target",
    width=None,
    height=None,
    xlim=None,
    ylim=None,
):
    """
    Scatter of error vs a chosen x-axis variable for a single model.

    Parameters
    ----------
    model_name : str
        Name of the model (key in MODEL_ERRORS).
    model_errors : dict
        MODEL_ERRORS dict.
    which : {"error", "pe", "ape"}
        - "error": y_pred - y_true (signed error)
        - "pe":    signed percentage error [%]
        - "ape":   absolute percentage error [%]
    x_source : {"target", "wsp", "ti"}
        - "target": x = true TARGET_COL
        - "wsp":    x = Wind Speed
        - "ti":     x = Turbulence Intensity
    xlim : tuple (xmin, xmax) or None
        If given, force x-axis range to this.
    ylim : tuple (ymin, ymax) or None
        If given, force y-axis range to this.
    """
    if model_name not in model_errors:
        print(f"Model '{model_name}' not found in MODEL_ERRORS.")
        return

    if which not in {"error", "pe", "ape"}:
        raise ValueError("which must be 'error', 'pe', or 'ape'")

    # y-values: error metric
    vals = np.asarray(model_errors[model_name][which])

    # x-values: choose source
    if x_source == "target":
        x_vals = np.asarray(y_test)
        x_label = f"True {TARGET_COL}"
    elif x_source == "wsp":
        x_vals = df_test["Wind Speed"].values
        x_label = "Wind Speed [m/s]"
    elif x_source == "ti":
        x_vals = df_test["Turbulence Intensity"].values
        x_label = "Turbulence Intensity [%]"
    else:
        raise ValueError("x_source must be 'target', 'wsp', or 'ti'")

    if which == "error":
        label_y = "Error (y_pred - y_true)"
    elif which == "pe":
        label_y = "Percentage error [%]"
    else:  # "ape"
        label_y = "Absolute percentage error [%]"

    title = f"{label_y} vs {x_label} – model: {model_name}"

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=x_vals,
        y=vals,
        mode="markers",
        name=model_name,
        marker=dict(size=6, opacity=0.6),
    ))

    # Reference line at 0 for signed quantities (error and pe)
    if which in {"error", "pe"}:
        y0 = 0.0
        x_min = float(np.min(x_vals))
        x_max = float(np.max(x_vals))
        fig.add_trace(go.Scatter(
            x=[x_min, x_max],
            y=[y0, y0],
            mode="lines",
            name="Zero line",
            line=dict(dash="dash"),
        ))

    layout_kwargs = dict(
        title=title,
        xaxis_title=x_label,
        yaxis_title=label_y,
        template="plotly_white",
    )

    # Apply global axis ranges if provided
    if xlim is not None:
        layout_kwargs["xaxis"] = dict(range=list(xlim))
    if ylim is not None:
        layout_kwargs["yaxis"] = dict(range=list(ylim))

    fig.update_layout(**layout_kwargs)

    if width is not None or height is not None:
        fig.update_layout(
            width=width if width is not None else DEFAULT_PLOT_WIDTH,
            height=height if height is not None else DEFAULT_PLOT_HEIGHT,
        )

    fig.show()


def plot_error_vs_x_for_models(
    model_names=None,
    which="error",
    x_source="target",
    width=None,
    height=None,
):
    """
    Loop wrapper: make one error-vs-x scatter per model,
    using the same x/y axis limits for all selected models.

    Parameters
    ----------
    model_names : list of str or None
        Which models to plot. If None, use all keys in MODEL_ERRORS.
    which : {"error", "pe", "ape"}
        Error type:
          - "error": signed error (y_pred - y_true)
          - "pe":    signed percentage error [%]
          - "ape":   absolute percentage error [%]
    x_source : {"target", "wsp", "ti"}
        What goes on x-axis: true target, wind speed, or TI.
    """
    if model_names is None:
        model_names = list(MODEL_ERRORS.keys())

    # --- compute global x and y ranges over all selected models ---

    # Build x_vals (depends on x_source)
    if x_source == "target":
        x_vals_global = np.asarray(y_test)
    elif x_source == "wsp":
        x_vals_global = df_test["Wind Speed"].values
    elif x_source == "ti":
        x_vals_global = df_test["Turbulence Intensity"].values
    else:
        raise ValueError("x_source must be 'target', 'wsp', or 'ti'")

    x_min = float(np.min(x_vals_global))
    x_max = float(np.max(x_vals_global))

    # Collect all errors for the chosen `which`
    all_err_vals = []
    for name in model_names:
        if name not in MODEL_ERRORS:
            print(f"Model '{name}' not in MODEL_ERRORS, skipping for range computation.")
            continue
        all_err_vals.append(MODEL_ERRORS[name][which])

    all_err_vals = np.concatenate(all_err_vals)
    y_min = float(np.min(all_err_vals))
    y_max = float(np.max(all_err_vals))

    # Add a small margin so points are not on the axis border
    x_margin = 0.05 * (x_max - x_min) if x_max > x_min else 1.0
    y_margin = 0.05 * (y_max - y_min) if y_max > y_min else 1.0

    xlim = (x_min - x_margin, x_max + x_margin)
    ylim = (y_min - y_margin, y_max + y_margin)

    # --- plot per model using common limits ---
    for name in model_names:
        if name not in MODEL_ERRORS:
            print(f"Model '{name}' not in MODEL_ERRORS, skipping.")
            continue

        print(f"\n=== Error vs {x_source} for model: {name} ({which}) ===")
        plot_error_vs_x_for_model(
            model_name=name,
            model_errors=MODEL_ERRORS,
            which=which,
            x_source=x_source,
            width=width,
            height=height,
            xlim=xlim,
            ylim=ylim,
        )


In [ ]:
# CELL 19: Run error vs X plots for all models

MODELS_FOR_ERR_X = list(MODEL_ERRORS.keys())   # or a subset

# Example 1: signed error vs true target
plot_error_vs_x_for_models(
    model_names=MODELS_FOR_ERR_X,
    which="pe",  #  "ape" for absolute percentage error or "error" for signed error or "pe" for percentage error
    x_source="target", #or "target" "wsp" or "ti"
    width=DEFAULT_PLOT_WIDTH ,
    height=DEFAULT_PLOT_HEIGHT,
)


## 11. Error patterns over the input space: scatter in (WS, TI)

Here we map the errors back onto the **(Wind Speed, TI)** plane:

- Each test point is plotted at its (WS, TI) coordinates,
- Colour shows the chosen error metric (`error`, `pe`, or `ape`).

We generate one scatter plot per model, using a **common colour scale**:

- Red/blue regions (for signed error) show where the model overpredicts
  or underpredicts.
- For `ape` we only see the magnitude of the relative error.

These plots help identify regions of the operating space where each model is
strong or weak.


In [ ]:
# CELL 20: Scatter of error over input space (WS vs TI)

def plot_error_over_inputs_scatter_for_model(
    model_name,
    model_errors,
    df_test,
    which="error",
    width=None,
    height=None,
    vmin=None,
    vmax=None,
):
    """
    2D scatter of error over (Wind Speed, Turbulence Intensity) for one model.

    - x-axis: Wind Speed
    - y-axis: Turbulence Intensity
    - color: chosen error metric

    Parameters
    ----------
    model_name : str
        Name of the model (key in MODEL_ERRORS).
    model_errors : dict
        MODEL_ERRORS dict.
    df_test : pd.DataFrame
        Test set with columns "Wind Speed" and "Turbulence Intensity".
    which : {"error", "pe", "ape"}
        "error" = y_pred - y_true
        "pe"    = signed percentage error [%]
        "ape"   = absolute percentage error [%]
    vmin, vmax : float or None
        Optional fixed color scale limits. If None, auto from data.
    """
    if model_name not in model_errors:
        print(f"Model '{model_name}' not found in MODEL_ERRORS.")
        return

    if which not in {"error", "pe", "ape"}:
        raise ValueError("which must be 'error', 'pe', or 'ape'")

    err_vals = np.asarray(model_errors[model_name][which])
    wsp = df_test["Wind Speed"].values
    ti  = df_test["Turbulence Intensity"].values

    if which == "error":
        cbar_label = "Error (y_pred - y_true)"
        colorscale = "RdBu"
    elif which == "pe":
        cbar_label = "Percentage error [%]"
        colorscale = "RdBu"
    else:
        cbar_label = "Absolute percentage error [%]"
        colorscale = "Viridis"

    if vmin is None:
        vmin = float(np.min(err_vals))
    if vmax is None:
        vmax = float(np.max(err_vals))

    fig = go.Figure()

    fig.add_trace(go.Scattergl(
        x=wsp,
        y=ti,
        mode="markers",
        name=model_name,
        marker=dict(
            size=7,
            opacity=0.7,
            color=err_vals,
            cmin=vmin,
            cmax=vmax,
            colorscale=colorscale,
            colorbar=dict(title=cbar_label),
        ),
    ))

    title = f"{which} over (WS, TI) – model: {model_name}, target: {TARGET_COL}"

    fig.update_layout(
        title=title,
        xaxis_title="Wind Speed [m/s]",
        yaxis_title="Turbulence Intensity [%]",
        template="plotly_white",
    )

    if width is not None or height is not None:
        fig.update_layout(
            width=width if width is not None else DEFAULT_PLOT_WIDTH,
            height=height if height is not None else DEFAULT_PLOT_HEIGHT,
        )

    fig.show()


def plot_error_over_inputs_scatter_for_models(
    model_names=None,
    model_errors=None,
    df_test=None,
    which="error",
    width=None,
    height=None,
):
    """
    Wrapper: one 2D error scatter per model, with a common color range.

    Parameters
    ----------
    model_names : list of str or None
        Models to include. If None, use all keys in MODEL_ERRORS.
    model_errors : dict
        MODEL_ERRORS dict.
    df_test : pd.DataFrame
        Test set.
    which : {"error", "pe", "ape"}
        Error metric to visualize.
    """
    if model_errors is None or df_test is None:
        print("model_errors and df_test must be provided.")
        return

    if model_names is None:
        model_names = list(model_errors.keys())

    # Determine global vmin/vmax across selected models for consistent color scale
    all_vals = []
    for name in model_names:
        if name not in model_errors:
            print(f"Model '{name}' not in MODEL_ERRORS, skipping for range computation.")
            continue
        all_vals.append(model_errors[name][which])

    all_vals = np.concatenate(all_vals)
    vmin = float(np.min(all_vals))
    vmax = float(np.max(all_vals))

    for name in model_names:
        if name not in model_errors:
            print(f"Model '{name}' not in MODEL_ERRORS, skipping.")
            continue

        print(f"\n=== Error over inputs (scatter) for model: {name}, metric: {which} ===")
        plot_error_over_inputs_scatter_for_model(
            model_name=name,
            model_errors=model_errors,
            df_test=df_test,
            which=which,
            width=width,
            height=height,
            vmin=vmin,
            vmax=vmax,
        )


In [ ]:
# CELL 20: Run error-over-inputs scatter plots

MODELS_FOR_ERR_SCATTER = list(MODEL_ERRORS.keys())  # or subset

# Choose which error metric to visualize: "error", "pe", or "ape"
ERROR_METRIC_FOR_INPUT_SCATTER = "pe"

plot_error_over_inputs_scatter_for_models(
    model_names=MODELS_FOR_ERR_SCATTER,
    model_errors=MODEL_ERRORS,
    df_test=df_test,
    which=ERROR_METRIC_FOR_INPUT_SCATTER,
    width=DEFAULT_PLOT_WIDTH,
    height=DEFAULT_PLOT_HEIGHT,
)


## 12. Binned error heatmaps over (WS, TI)

Finally, we summarise the error behaviour over the (WS, TI) space using
**2D binned heatmaps**:

- The (WS, TI) domain is divided into bins,
- For each bin we compute the **mean error** (or percentage error),
- The result is shown as a 2D colour map.

Each model gets one heatmap, with a **shared colour scale** so that we can
compare models directly.

These plots are useful for:

- Identifying regions where a model is consistently biased,
- Seeing where relative errors are systematically higher,
- Discussing where additional training data or a different surrogate
  might be needed.


In [ ]:
# CELL 21: 2D binned heatmap of mean error over (WS, TI)

def compute_binned_mean_error(
    wsp,
    ti,
    err_vals,
    n_wsp_bins=12,
    n_ti_bins=8,
):
    """
    Bin (wsp, ti) space and compute mean error per bin.

    Returns
    -------
    H_mean : 2D array (n_ti_bins, n_wsp_bins)
        Mean error in each 2D bin. NaN where no points.
    wsp_edges, ti_edges : 1D arrays
        Bin edges for WS and TI.
    """
    wsp = np.asarray(wsp)
    ti  = np.asarray(ti)
    err_vals = np.asarray(err_vals)

    wsp_min, wsp_max = np.min(wsp), np.max(wsp)
    ti_min, ti_max   = np.min(ti), np.max(ti)

    wsp_edges = np.linspace(wsp_min, wsp_max, n_wsp_bins + 1)
    ti_edges  = np.linspace(ti_min, ti_max, n_ti_bins + 1)

    # Assign each point to a bin
    wsp_idx = np.digitize(wsp, wsp_edges) - 1
    ti_idx  = np.digitize(ti,  ti_edges)  - 1

    H_sum  = np.zeros((n_ti_bins, n_wsp_bins), dtype=float)
    H_count = np.zeros_like(H_sum)

    for w_i, t_i, e in zip(wsp_idx, ti_idx, err_vals):
        if 0 <= w_i < n_wsp_bins and 0 <= t_i < n_ti_bins:
            H_sum[t_i, w_i] += e
            H_count[t_i, w_i] += 1

    # Avoid division by zero
    with np.errstate(invalid="ignore", divide="ignore"):
        H_mean = H_sum / H_count
    H_mean[H_count == 0] = np.nan

    return H_mean, wsp_edges, ti_edges


def plot_error_heatmap_for_model(
    model_name,
    model_errors,
    df_test,
    which="error",
    n_wsp_bins=12,
    n_ti_bins=8,
    vmin=None,
    vmax=None,
    width=None,
    height=None,
):
    """
    2D heatmap of binned mean error over (WS, TI) for one model.
    """
    if model_name not in model_errors:
        print(f"Model '{model_name}' not in MODEL_ERRORS.")
        return

    if which not in {"error", "pe", "ape"}:
        raise ValueError("which must be 'error', 'pe', or 'ape'")

    err_vals = np.asarray(model_errors[model_name][which])
    wsp = df_test["Wind Speed"].values
    ti  = df_test["Turbulence Intensity"].values

    H_mean, wsp_edges, ti_edges = compute_binned_mean_error(
        wsp, ti, err_vals,
        n_wsp_bins=n_wsp_bins,
        n_ti_bins=n_ti_bins,
    )

    wsp_centers = 0.5 * (wsp_edges[:-1] + wsp_edges[1:])
    ti_centers  = 0.5 * (ti_edges[:-1]  + ti_edges[1:])

    if which == "error":
        cbar_label = "Mean error (y_pred - y_true)"
        colorscale = "RdBu"
    elif which == "pe":
        cbar_label = "Mean percentage error [%]"
        colorscale = "RdBu"
    else:
        cbar_label = "Mean absolute percentage error [%]"
        colorscale = "Viridis"

    if vmin is None:
        vmin = np.nanmin(H_mean)
    if vmax is None:
        vmax = np.nanmax(H_mean)

    fig = go.Figure(data=go.Heatmap(
        x=wsp_centers,
        y=ti_centers,
        z=H_mean,
        zmin=vmin,
        zmax=vmax,
        colorscale=colorscale,
        colorbar=dict(title=cbar_label),
    ))

    title = f"Binned mean {which} over (WS, TI) – model: {model_name}, target: {TARGET_COL}"

    fig.update_layout(
        title=title,
        xaxis_title="Wind Speed [m/s]",
        yaxis_title="Turbulence Intensity [%]",
        template="plotly_white",
    )

    if width is not None or height is not None:
        fig.update_layout(
            width=width if width is not None else DEFAULT_PLOT_WIDTH,
            height=height if height is not None else DEFAULT_PLOT_HEIGHT,
        )

    fig.show()


def plot_error_heatmaps_for_models(
    model_names=None,
    model_errors=None,
    df_test=None,
    which="error",
    n_wsp_bins=12,
    n_ti_bins=8,
    width=None,
    height=None,
):
    """
    Wrapper: one heatmap per model, with a common color scale.
    """
    if model_errors is None or df_test is None:
        print("model_errors and df_test must be provided.")
        return

    if model_names is None:
        model_names = list(model_errors.keys())

    # Determine global vmin/vmax across selected models
    all_means = []
    for name in model_names:
        if name not in model_errors:
            print(f"Model '{name}' not in MODEL_ERRORS, skipping for range computation.")
            continue

        err_vals = np.asarray(model_errors[name][which])
        wsp = df_test["Wind Speed"].values
        ti  = df_test["Turbulence Intensity"].values

        H_mean, _, _ = compute_binned_mean_error(
            wsp, ti, err_vals,
            n_wsp_bins=n_wsp_bins,
            n_ti_bins=n_ti_bins,
        )
        all_means.append(H_mean)

    H_all = np.stack(all_means, axis=0)
    vmin = float(np.nanmin(H_all))
    vmax = float(np.nanmax(H_all))

    for name in model_names:
        if name not in model_errors:
            print(f"Model '{name}' not in MODEL_ERRORS, skipping.")
            continue

        print(f"\n=== Error heatmap for model: {name}, metric: {which} ===")
        plot_error_heatmap_for_model(
            model_name=name,
            model_errors=model_errors,
            df_test=df_test,
            which=which,
            n_wsp_bins=n_wsp_bins,
            n_ti_bins=n_ti_bins,
            vmin=vmin,
            vmax=vmax,
            width=width,
            height=height,
        )


In [ ]:
# CELL 21: Run 2D error heatmaps over WS–TI

MODELS_FOR_ERR_HEATMAP = list(MODEL_ERRORS.keys())  # or subset

ERROR_METRIC_FOR_HEATMAP = "pe"  # "error", "pe", or "ape"

plot_error_heatmaps_for_models(
    model_names=MODELS_FOR_ERR_HEATMAP,
    model_errors=MODEL_ERRORS,
    df_test=df_test,
    which=ERROR_METRIC_FOR_HEATMAP,
    n_wsp_bins=6,
    n_ti_bins=6,
    width=DEFAULT_PLOT_WIDTH,
    height=DEFAULT_PLOT_HEIGHT,
)
